# 模块 10：Transformer

本 Notebook 组合位置编码、LayerNorm、Self-Attention 和 FeedForward，手写一个最小 Transformer Encoder Block。

## 1. 理论背景

Transformer 的核心是让序列中任意位置通过 Self-Attention 直接交互，再用前馈网络独立更新每个 token 的表示。

## 1.1 实际作用（用途）
- 是现代大语言模型、机器翻译、文本摘要、代码生成、语音识别和视觉 Transformer 的核心架构。
- 能并行处理序列，适合在大规模语料和硬件加速器上扩展训练。
- Encoder 偏理解任务，Decoder 偏生成任务，Decoder-only 是许多 LLM 的主体结构。

## 2. 数学推导

$$x_1 = LayerNorm(x + SelfAttention(x))$$

$$y = LayerNorm(x_1 + FFN(x_1))$$

位置编码使用经典正弦/余弦公式：

$$PE(pos,2i)=sin(pos/10000^{2i/d})$$
$$PE(pos,2i+1)=cos(pos/10000^{2i/d})$$

In [ ]:
import numpy as np

def positional_encoding(length, d_model):
    # positions 是 token 位置，dims 是偶数维索引；二者广播得到所有角度。
    positions = np.arange(length)[:, None]
    dims = np.arange(0, d_model, 2)[None, :]
    angles = positions / np.power(10000, dims / d_model)
    pe = np.zeros((length, d_model), dtype=np.float64)
    pe[:, 0::2] = np.sin(angles)
    pe[:, 1::2] = np.cos(angles[:, : pe[:, 1::2].shape[1]])
    return pe

def layer_norm(x, eps=1e-5):
    # LayerNorm 在最后一个维度上归一化，每个 token 单独计算均值和方差。
    mean = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(var + eps)

def softmax(x, axis=-1):
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=axis, keepdims=True)

def self_attention(x, wq, wk, wv, mask=None):
    # 输入 x 通过三组线性投影得到 Q/K/V。
    q, k, v = x @ wq, x @ wk, x @ wv
    scores = q @ k.T / np.sqrt(q.shape[-1])
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = softmax(scores, axis=-1)
    return weights @ v, weights

def feed_forward(x, w1, b1, w2, b2):
    # Transformer block 中的 FFN 是逐 token 应用的两层 MLP。
    hidden = np.maximum(0, x @ w1 + b1)
    return hidden @ w2 + b2


## 3. 代码思路

本章代码把 Transformer Encoder Block 拆成几个可独立理解的函数：

1. `positional_encoding` 生成正弦/余弦位置编码，并加到 token 表示上，让模型获得顺序信息。
2. `layer_norm` 对每个 token 的特征维度做归一化，稳定残差连接后的数值分布。
3. `self_attention` 把输入投影成 Q/K/V，计算注意力权重，再汇总 Value 得到上下文表示。
4. `feed_forward` 是逐 token 的两层 MLP，用于在 attention 之后进一步变换每个位置的表示。
5. 主流程按 `Self-Attention -> 残差 + LayerNorm -> FFN -> 残差 + LayerNorm` 组合成最小 Encoder Block，并额外演示 causal mask 的效果。

## 4. NumPy 手写实现

构造一个最小 Encoder Block。为了突出结构，权重使用固定随机种子。

In [ ]:
rng = np.random.default_rng(10)
tokens = ["从", "零", "实现", "网络"]
length, d_model, d_ff = len(tokens), 6, 12
# token 表示加上位置编码后，Self-Attention 才能感知顺序信息。
x = rng.normal(scale=0.2, size=(length, d_model)) + positional_encoding(length, d_model)

# 初始化 attention 和 feed-forward 的权重；真实训练中这些参数会被反向传播更新。
wq = rng.normal(scale=0.2, size=(d_model, d_model))
wk = rng.normal(scale=0.2, size=(d_model, d_model))
wv = rng.normal(scale=0.2, size=(d_model, d_model))
w1 = rng.normal(scale=0.2, size=(d_model, d_ff))
b1 = np.zeros(d_ff)
w2 = rng.normal(scale=0.2, size=(d_ff, d_model))
b2 = np.zeros(d_model)

# 一个最小 Encoder Block：Self-Attention -> 残差 + LayerNorm -> FFN -> 残差 + LayerNorm。
attn_out, attn_weights = self_attention(x, wq, wk, wv)
x1 = layer_norm(x + attn_out)
ffn_out = feed_forward(x1, w1, b1, w2, b2)
y = layer_norm(x1 + ffn_out)

print("attention weights:\n", np.round(attn_weights, 3))
print("encoder output shape:", y.shape)


## 5. Causal Mask

Decoder-only LLM 使用 causal mask，确保当前位置只能看见历史 token。

In [ ]:
# causal mask 用于 Decoder：第 t 个 token 只能看见 0..t 的历史位置。
causal_mask = np.tril(np.ones((length, length), dtype=bool))
_, masked_weights = self_attention(x, wq, wk, wv, mask=causal_mask)
print(np.round(masked_weights, 3))


## 6. PyTorch 数值验证

验证手写 LayerNorm 与 PyTorch `layer_norm` 一致。

In [ ]:
import torch

# 用 PyTorch 的 layer_norm 对照 NumPy 实现，确认归一化维度和 eps 设置一致。
torch_ln = torch.nn.functional.layer_norm(
    torch.tensor(x, dtype=torch.float64),
    normalized_shape=(d_model,),
    weight=None,
    bias=None,
    eps=1e-5,
)
assert np.allclose(layer_norm(x), torch_ln.numpy(), atol=1e-5)
print("NumPy LayerNorm matches PyTorch")
